In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import PIIMiddleware

load_dotenv()

# =================1.定义PIIMiddleware==========================
# 也可以添加多个PII中间件来检测不同类型的敏感信息
pii_middleware_email = PIIMiddleware(
    "email",
    strategy="mask",  # mask
    apply_to_input=False,
    apply_to_output=True
)

pii_middleware_phone = PIIMiddleware(
    "phone_number",
    # 使用自定义正则表达式
    detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
    strategy="mask",
    apply_to_input=True
)

# ====================2.在Agent中注册PIIMiddleware================
agent = create_agent(
    model="deepseek-flash",
    tools=[],
    middleware=[pii_middleware_email, pii_middleware_phone],
    system_prompt="需要过滤电话和email中的敏感信息"
)

# ======================3.测试=================================
response = agent.invoke({"messages":[HumanMessage(
    "我的邮箱是: huge@itcast.cn,我的电话是13698023405,你要记住这些信息，后续要用到。如果记住了请确认一遍。")]})

for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

我的邮箱是: huge@itcast.cn,我的电话是****3405,你要记住这些信息，后续要用到。如果记住了请确认一遍。
================================== Ai Message ==================================

已记住：邮箱已记录（h***@itcast.cn），电话已记录（****3405）。后续需要时请提示我使用。


In [6]:
#2 ModelFallbackMiddleware
from langchain.agents.middleware import ModelFallbackMiddleware

model_fallback_middleware = ModelFallbackMiddleware(
        "deepseek-v", #默认模型
        "deepseek-flash" #备用模型
)

agent = create_agent(
    model="deepseek-v",
    tools=[],
    system_prompt="模型报错反馈",
    middleware=[model_fallback_middleware]
)

response= agent.invoke({"messages":[HumanMessage("你好")]})

for token in response['messages']:
    token.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！很高兴见到你 😊

有什么我可以帮你的吗？无论是问题咨询、写作、编程、学习还是其他方面，都可以告诉我。


In [7]:
#HumanInTheLoop，简称为HITL，其作用是让人工介入到Agent执行流程中，在执行工具调用前暂停，等待人工确认。

#发邮件前需要人工审核案例

emails = [
        {
            "subject": "周末见个面？",
            "content": """
                嗨 虎哥，
                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

                祝好，简
            """,
            "from": "jane@itcast.cn",
            "status": "unread"
        },
        {
            "subject": "周五会议",
            "content": """
                嗨 虎哥，
                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

                祝好，小李
            """,
            "from": "lixiaolong@itcast.cn",
            "status": "checked"
        }
    ]

In [8]:
from langchain.tools import tool

# 定义工具，收取和发送邮件
@tool
def check_inbox() -> str:
    """查看收件箱中的邮件."""

    # 模拟收件箱邮件
    return emails


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送回复邮件"""
    return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"

In [9]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

# 创建人工审核中间件
human_in_loop_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": {#需要人工确认的工具的名字
            "description": "请确认邮件内容",
            "allowed_decisions": ["approve", "reject", "edit"]
        }
    }
)

In [10]:
from langgraph.checkpoint.memory import InMemorySaver
agent_with_hitl = create_agent(
    model="deepseek-flash",
    tools=[check_inbox, send_email],
    middleware=[human_in_loop_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="你是一个邮件助手"
)

config = {"configurable": {"thread_id": "1"}}
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我查看邮件")]},
    config=config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看收件箱中的邮件。
Tool Calls:
  check_inbox (call_00_DNfhqsaZWeLWOKPfvgeB9446)
 Call ID: call_00_DNfhqsaZWeLWOKPfvgeB9446
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱里目前有两封邮件：

**1. 📬 周末见个面？**（未读）
- **发件人**：jane@itcast.cn
- **内容**：简说她下周会去城里，想约您一起喝杯咖啡，看看有没有机会见面。

**2. 📬 周五会议**（已读）
- **发件人**：li

In [11]:
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我回复Jane，很高兴她能来，问问她具体见面时间")]},
    config=config,
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看收件箱中的邮件。
Tool Calls:
  check_inbox (call_00_DNfhqsaZWeLWOKPfvgeB9446)
 Call ID: call_00_DNfhqsaZWeLWOKPfvgeB9446
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱里目前有两封邮件：

**1. 📬 周末见个面？**（未读）
- **发件人**：jane@itcast.cn
- **内容**：简说她下周会去城里，想约您一起喝杯咖啡，看看有没有机会见面。

**2. 📬 周五会议**（已读）
- **发件人**：li

In [12]:
print(response)

{'messages': [HumanMessage(content='帮我查看邮件', additional_kwargs={}, response_metadata={}, id='efe948f6-cd23-496f-a816-7bca2e90ee07'), AIMessage(content='我来帮您查看收件箱中的邮件。', additional_kwargs={'refusal': None, 'reasoning_content': 'The user wants me to check their inbox. Let me call the check_inbox tool.'}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 350, 'total_tokens': 400, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 18, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 222}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': 'acc3be19-567d-4dbb-84a1-8fbdce8fc8de', 'finish_reason': 'tool_calls', 'logprobs': None},

In [13]:
#如果用户拒绝Agent调用工具，我们需要再次调用Agent，并通过Command来告知Agent用户选择了reject，并且告知拒绝原因。
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",            # 人工确认的决定，是reject
                    "message": "语气太书面了，Jane是我的女朋友！" # 请求被拒绝的原因
                }
            ]
        }
    ),
    config=config  # 相同的thread_id以恢复暂停的对话
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看收件箱中的邮件。
Tool Calls:
  check_inbox (call_00_DNfhqsaZWeLWOKPfvgeB9446)
 Call ID: call_00_DNfhqsaZWeLWOKPfvgeB9446
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱里目前有两封邮件：

**1. 📬 周末见个面？**（未读）
- **发件人**：jane@itcast.cn
- **内容**：简说她下周会去城里，想约您一起喝杯咖啡，看看有没有机会见面。

**2. 📬 周五会议**（已读）
- **发件人**：li

In [14]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config  # 相同的thread_id以恢复暂停的对话
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看收件箱中的邮件。
Tool Calls:
  check_inbox (call_00_DNfhqsaZWeLWOKPfvgeB9446)
 Call ID: call_00_DNfhqsaZWeLWOKPfvgeB9446
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱里目前有两封邮件：

**1. 📬 周末见个面？**（未读）
- **发件人**：jane@itcast.cn
- **内容**：简说她下周会去城里，想约您一起喝杯咖啡，看看有没有机会见面。

**2. 📬 周五会议**（已读）
- **发件人**：li

In [15]:
#自定义中间件,装饰器加函数形式
#nodule style

from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import after_model


# 1.定义自定义state，记录模型调用次数
class CustomState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: int  # 模型调用次数


# 2.定义中间件
@after_model(state_schema=CustomState)
def increment_counter(
    state: CustomState, #nodule style必须有
    runtime: Runtime #nodule style必须有
):

    # 读取State中的count值
    current_count = state.get("model_call_count", 0)
    # 返回dict，其中的key/value是要更新的State值
    return {"model_call_count": current_count + 1}


# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-chat",
    middleware=[increment_counter],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体
response = agent.invoke(
    {"messages": [HumanMessage("Hello")],},
    config
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='af2531f7-3c7d-457f-a7bc-0cbe3a5a5173'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 5, 'total_tokens': 14, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '6933254e-b420-4b76-b0bd-88bf145e1708', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17778-3af1-722b-a5b5-a6c5dad964a6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 9, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}, 'output_token_details':

In [16]:
# 调用智能体
response = agent.invoke(
    {"messages": [HumanMessage("讲个笑话吧")],},
    config
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='af2531f7-3c7d-457f-a7bc-0cbe3a5a5173'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 5, 'total_tokens': 14, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '6933254e-b420-4b76-b0bd-88bf145e1708', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17778-3af1-722b-a5b5-a6c5dad964a6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 9, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}, 'output_token_details':

In [17]:
#wrap style 中间件

from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from typing import Any, Callable


@wrap_model_call
def retry_model(
    request: ModelRequest, #wrap style必须有
    handler: Callable[[ModelRequest], ModelResponse], #wrap style必须有
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request) # 这一步就是调用模型
        except Exception as e:
            print(f"Retry {attempt + 1}/3 after error: {e}")
            if attempt == 2:
                raise
    return None


agent = create_agent(
    model="deepseek-chat",
    middleware=[retry_model]
)

In [18]:
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='8f1014a0-d588-4d69-8782-31d9cb5455bc'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 5, 'total_tokens': 14, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': 'fbd1f052-bb8c-4e8f-91f5-af1cb801cb2d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17778-4734-77ad-801d-e97e785e8bf0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 9, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}, 'output_token_details':

In [19]:
# 动态修改请求参数

#例如，我们定义中间件，可以根据context中用户的选择来开启或关闭模型的思考模式。

from langchain.agents.middleware import wrap_model_call
from collections.abc import Callable
from pydantic.dataclasses import dataclass

# 定义一个context，记录用户是否开启思考模式
@dataclass
class UserContext:
    thinking:bool = False

#自定义中间件
@wrap_model_call
def dynamic_model_settings(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:

    #获取上下文参数thinking，是否开启思考模式
    thinking = request.runtime.context.thinking

    #如果Thinking为true，则开启思考模式，否则关闭思考模式
    thinking_type = "enabled" if thinking else "disabled"
    model_settings = {"extra_body":{"thinking":{"type":thinking_type}}}

    #覆盖request中的model参数
    modified_quest = request.override(model_settings= model_settings)
    return handler(modified_quest)

# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-flash",
    middleware=[dynamic_model_settings],
    context_schema=UserContext
)

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    context=UserContext(thinking=False) # 通过Context控制思考模式开关
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='3044f504-37dd-41b3-8460-6bad616cc281'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 5, 'total_tokens': 14, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '7705aead-b10b-4684-bc35-47a596bf3e73', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17778-4a6c-72b0-8ac8-d5146b1bf4b1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 9, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}, 'output_token_details':

In [20]:
#middleware jump_to跳转

from langgraph.runtime import Runtime
from typing import NotRequired, Any
from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    model_call_count: int

# 2.定义中间件
class ModelCallLimitMiddleware(AgentMiddleware[CustomAgentState]):

    def __init__(self, max_limit: int = 10):
        super().__init__()
        self.max_limit = max_limit

    # after_model hook，模型调用次数计数
    def after_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        current_count = state.get("model_call_count", 0)
        return {"model_call_count": current_count + 1}

    # before_model hook，校验模型调用次数是否超限
    @hook_config(can_jump_to=["end"])
    def before_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        # 获取模型调用次数
        current_count = state.get("model_call_count", 0) + 1
        # 判断是否超限
        print(f"当前模型调用次数：{current_count}/{self.max_limit}")
        if current_count > self.max_limit:
            return {
                "jump_to": "end",
                "messages": AIMessage(f"模型调用次数超过最大限制：{self.max_limit}！")
            }
        return None

# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-chat",
    middleware=[ModelCallLimitMiddleware(max_limit=2)],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
}, config)

print(response)

当前模型调用次数：1/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='c7792440-caf3-42e0-92c7-37aa8bce2746'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 5, 'total_tokens': 14, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '66cce0ff-b3ca-45cc-b181-ece0b436d194', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17778-4d7e-7344-9bd9-c84c75361f87-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 9, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}, 'output_to

In [21]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("say hi")],
    "model_call_count": 0,
}, config)

print(response)

当前模型调用次数：1/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='c7792440-caf3-42e0-92c7-37aa8bce2746'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 5, 'total_tokens': 14, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '66cce0ff-b3ca-45cc-b181-ece0b436d194', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17778-4d7e-7344-9bd9-c84c75361f87-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 9, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}, 'output_to

In [ ]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("say a joke")],
    "model_call_count": 0,
}, config)

print(response)

当前模型调用次数：1/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='c7792440-caf3-42e0-92c7-37aa8bce2746'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 5, 'total_tokens': 14, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '66cce0ff-b3ca-45cc-b181-ece0b436d194', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17778-4d7e-7344-9bd9-c84c75361f87-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 9, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}, 'output_to

: 